# 📝 Day 8 Assignments — Relationships & Integration

---

Four tasks plus a bonus. By the end you'll have a working FastAPI app where users own posts, posts can be tagged, and `DELETE /users/{id}` cleans everything up via cascade.

In [ ]:
!pip install sqlalchemy fastapi uvicorn pydantic httpx

## Task 1 — `User` ↔ `Post` one-to-many

**Problem:**

1. Define `User` and `Post` models with a one-to-many relationship.
   - `User.posts` → list of `Post`
   - `Post.author` → back to `User`
   - Use `cascade="all, delete-orphan"` on `User.posts`.
2. Create one user with three posts in a single `session.add(user)` call.
3. Print `user.name` and `[p.title for p in user.posts]`.

**Expected output:**
```
Alice ['First', 'Second', 'Third']
```

💡 **Hint:** `Post(title=...)` instances passed via `User(posts=[...])` get inserted automatically when you commit the user.

In [ ]:
from __future__ import annotations
from sqlalchemy import create_engine, ForeignKey, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, sessionmaker

engine = create_engine("sqlite:///./day8_assignments.db", echo=False)

class Base(DeclarativeBase):
    pass

class User(Base):
    __tablename__ = "users"
    # TODO: id (PK), name, email (unique)
    # TODO: posts relationship with cascade="all, delete-orphan"

class Post(Base):
    __tablename__ = "posts"
    # TODO: id (PK), title, author_id (FK -> users.id)
    # TODO: author relationship with back_populates="posts"

Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)
SessionLocal = sessionmaker(bind=engine)

with SessionLocal() as session:
    # TODO: create Alice with three posts in one go, commit, and print
    pass

## Task 2 — `POST /users` with Pydantic schemas

**Problem:**

1. Define `UserCreate` (request) and `UserOut` (response) Pydantic models — keep them **separate** from the SQLAlchemy `User`.
2. Build a FastAPI app with `POST /users` that inserts a row and returns `UserOut`.
3. Use `TestClient` to send a valid payload and print the response.

💡 **Hint:** put `model_config = ConfigDict(from_attributes=True)` on `UserOut` so FastAPI can serialize the SQLAlchemy object directly.

In [ ]:
from fastapi import FastAPI, Depends, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, ConfigDict
from sqlalchemy.orm import Session

class UserCreate(BaseModel):
    # TODO
    pass

class UserOut(BaseModel):
    # TODO: id, name, email + model_config = ConfigDict(from_attributes=True)
    pass

def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

app = FastAPI()

@app.post("/users", response_model=UserOut, status_code=201)
def create_user(payload: UserCreate, db: Session = Depends(get_db)):
    # TODO: build User, add, commit, refresh, return
    pass

client = TestClient(app)
# TODO: POST a valid user and print the response

## Task 3 — `GET /users/{id}` and `GET /users/{id}/posts`

**Problem:** Add two more endpoints to the app from Task 2:

- `GET /users/{id}` → return one user (404 if missing).
- `GET /users/{id}/posts` → return the user's posts as `list[PostOut]`.

Then with `TestClient`:

1. POST a new user.
2. POST a couple of posts for that user (you'll need a `POST /users/{id}/posts` endpoint too).
3. Hit `GET /users/{id}/posts` and confirm the list.

💡 **Hint:** define `PostCreate` and `PostOut` schemas the same way you did `UserCreate` / `UserOut`.

In [ ]:
class PostCreate(BaseModel):
    # TODO
    pass

class PostOut(BaseModel):
    # TODO: id, title, author_id + from_attributes=True
    pass

# TODO: @app.get("/users/{user_id}")
# TODO: @app.post("/users/{user_id}/posts")
# TODO: @app.get("/users/{user_id}/posts")

# TODO: TestClient flow — create user, create 2 posts, GET the posts, print

## Task 4 — Cascade delete

**Problem:**

1. Add `DELETE /users/{id}` that deletes the user.
2. After deletion, `select(Post)` should return **zero** rows — the cascade should have wiped the posts too.
3. Verify with `TestClient` + an assertion.

💡 **Hint:** the cascade is declared on the parent's `relationship(...)`. SQLAlchemy issues the child DELETEs for you when you `session.delete(user)`.

In [ ]:
# TODO: @app.delete("/users/{user_id}", status_code=204)

# TODO: TestClient — create user + posts, DELETE the user, assert no posts remain

## 🎁 Bonus — Many-to-many tags

**Problem:**

1. Add a `Tag` model and a `post_tags` association table linking `Post` ↔ `Tag`.
2. Add `GET /tags/{tag_id}/posts` returning every post carrying that tag.
3. Demo with `TestClient`: create a tag, attach it to two posts, hit the endpoint.

💡 **Hint:** `secondary=post_tags` on both `Post.tags` and `Tag.posts`. The association table uses `Column("...", ForeignKey("..."), primary_key=True)` for both columns.

In [ ]:
from sqlalchemy import Table, Column

# TODO: post_tags association table

class Tag(Base):
    __tablename__ = "tags"
    # TODO: id, name (unique)
    # TODO: posts relationship via secondary=post_tags

# TODO: attach Post.tags = relationship("Tag", secondary=post_tags, back_populates="posts")
# TODO: recreate tables
# TODO: @app.get("/tags/{tag_id}/posts", response_model=list[PostOut])
# TODO: TestClient demo

---

✅ **Done!** You now have a real database-backed FastAPI app with relationships, cascade, and a clean schema/model split. Next stop: deploying it.